# Week 60, Day 6: Forge

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w060d6.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
def make():
    n = 0
    def bump():
        nonlocal n
        n += 2
        return n
    return bump

b = make()
print(b(), b(), make()())

**Predict before you run:** what will this print?



In [ ]:
fs = [lambda: k for k in "ab"]
gs = [lambda k=k: k for k in "ab"]
print([f() for f in fs], [g() for g in gs])

**Predict before you run:** what will this print?



In [ ]:
def deco(func):
    print("decorating")
    def wrapper():
        return func() * 2
    return wrapper

@deco
def two():
    return 2

print(two(), two())

**Predict before you run:** what will this print?



In [ ]:
def times(n):
    def deco(f):
        return lambda x: f(x) * n
    return deco

@times(3)
@times(2)
def ident(x):
    return x

print(ident(5))

**Predict before you run:** what will this print?



In [ ]:
from functools import partial, reduce
print(partial(int, base=8)("17"), reduce(lambda a, b: a + b, "xyz", "!"))

### Exercise: Tracing recursion

Write a decorator `traced(func)` that prints each call and its result, indented by two spaces per level of recursion, using a closure variable for the current depth (with `nonlocal`, restoring it even if the call raises). Each call prints `"{indent}{name}({args})"` before and `"{indent}-> {result}"` after, with arguments shown by `repr` and joined by `", "`.

In [ ]:
import functools

def traced(func):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_traced():\n    \"nonlocal depth\"\n    @traced\n    def fact(n):\n        return 1 if n <= 1 else n * fact(n - 1)\n    import io, contextlib\n    buf = io.StringIO()\n    with contextlib.redirect_stdout(buf):\n        fact(3)\n    lines = buf.getvalue().strip().splitlines()\n    assert lines == [\"fact(3)\", \"  fact(2)\", \"    fact(1)\", \"    -> 1\", \"  -> 2\", \"-> 6\"], lines\n    assert fact.__name__ == \"fact\"")

### Exercise: A generator with a memory

Write `make_numberer(start=1)`: returns a **generator function** `numbered(items)` that yields `(n, item)` pairs, where the numbering **continues** from where the previous call left off (a closure with `nonlocal` holding the next number, shared by every call of `numbered`).

In [ ]:
def make_numberer(start=1):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import types\ndef test_numberer():\n    \"closures and generators\"\n    numbered = make_numberer()\n    assert list(numbered(\"ab\")) == [(1, \"a\"), (2, \"b\")] and list(numbered([\"c\"])) == [(3, \"c\")]\n    other = make_numberer(10)\n    assert list(other(\"x\")) == [(10, \"x\")] and list(numbered(\"d\")) == [(4, \"d\")]\n    assert isinstance(numbered(\"z\"), types.GeneratorType)")

### Exercise: Forge: a plugin host

Write `PluginHost`, `timed` and `expects` to the specification above.

In [ ]:
import functools

class PluginHost:
    def __init__(self):
        ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def make():\n    host = PluginHost()\n\n    @host.plugin\n    def word_count(text):\n        return len(text.split())\n\n    @host.plugin(name=\"shout\", priority=5)\n    def loud(text):\n        return text.upper()\n\n    @host.plugin(priority=5)\n    def chars(text):\n        return len(text)\n\n    return host, word_count, loud\n\ndef test_register():\n    \"registering plugins\"\n    host, word_count, loud = make()\n    assert host.plugins() == [\"chars\", \"shout\", \"word_count\"]\n    assert word_count(\"a b\") == 2 and loud(\"x\") == \"X\", \"decorating returns the function\"\n    try:\n        @host.plugin\n        def chars(text):\n            return 0\n        assert False\n    except ValueError:\n        pass\n    try:\n        host.hook(\"during\")\n        assert False\n    except ValueError:\n        pass\n\ndef test_run():\n    \"running with hooks\"\n    host, _, _ = make()\n    seen = []\n    @host.hook(\"before\")\n    def note(name, args):\n        seen.append((\"before\", name, args))\n    @host.hook(\"after\")\n    def done(name, result):\n        seen.append((\"after\", name, result))\n    assert host.run(\"shout\", \"hi\") == \"HI\"\n    assert seen == [(\"before\", \"shout\", (\"hi\",)), (\"after\", \"shout\", \"HI\")]\n    assert host.run_all(\"the cat sat\") == {\"chars\": 11, \"shout\": \"THE CAT SAT\", \"word_count\": 3}\n    assert list(host.run_all(\"x\")) == [\"chars\", \"shout\", \"word_count\"]\n    assert host.stats() == {\"word_count\": 2, \"shout\": 3, \"chars\": 2}\n    try:\n        host.run(\"nope\")\n        assert False\n    except KeyError:\n        pass\n\ndef test_disable():\n    \"enable and disable\"\n    host, _, _ = make()\n    host.disable(\"shout\")\n    assert \"shout\" not in host.run_all(\"a\") and host.stats()[\"shout\"] == 0\n    try:\n        host.run(\"shout\", \"a\")\n        assert False\n    except RuntimeError:\n        pass\n    host.enable(\"shout\")\n    assert host.run(\"shout\", \"a\") == \"A\"\n    try:\n        host.disable(\"ghost\")\n        assert False\n    except KeyError:\n        pass\n\ndef test_decorators():\n    \"decorators on plugins\"\n    now = [0.0]\n    def clock():\n        now[0] += 0.5\n        return now[0]\n    host = PluginHost()\n    @host.plugin\n    @timed(clock)\n    @expects(str, int)\n    def repeat(text, times):\n        return text * times\n    assert host.run(\"repeat\", \"ab\", 2) == \"abab\" and repeat.last == 0.5 and repeat.__name__ == \"repeat\"\n    try:\n        host.run(\"repeat\", \"ab\", \"2\")\n        assert False\n    except TypeError as e:\n        assert str(e) == \"repeat: argument 1 must be int\"\n    assert repeat.last == 0.5 and host.stats() == {\"repeat\": 1}")

> **Decorators as an interface.** The plugin authors never call `host.register(...)`; they write `@host.plugin` above an ordinary function. That's why frameworks lean on decorators: registration sits right next to the code it registers, plugins stay plain functions you can call and test directly (the decorator returns them unchanged), and settings read naturally (`@host.plugin(priority=5)`). Flask's `@app.route("/")`, pytest's `@pytest.fixture` and click's `@click.command()` all follow this shape, and all of them are closures and factories like the ones you wrote this week.

### Exercise: Forge bonus: cleaning recipes

A data team writes text-cleaning recipes as strings like `"strip|lower|squash|replace:tea:coffee|cut:12"`. Build the machinery:

- `steps = {}` and a registering decorator factory `step(name)` that stores a function under that name and returns it unchanged. Register `strip`, `lower`, `squash` (collapse runs of whitespace to single spaces and trim the ends), `replace(text, old, new)` and `cut(text, n)` (the first n characters; `n` may arrive as a string).
- `compose(*funcs)`: a function applying them left to right, with `functools.reduce` (no functions: the identity).
- `build(spec)`: split the recipe on `"|"`; each part is a step name followed by `:`-separated arguments, which are passed to the step after the text, as strings, by name (use `functools.partial` with the step's parameter names after the first, read from `func.__code__`). Unknown steps raise `KeyError`. Return the composed function.
- `describe(spec)`: a readable version: `"strip then replace(a, b) then cut(3)"`.

New steps registered later with `@step(...)` must work in `build` straight away.

In [ ]:
from functools import partial, reduce

steps = {}

def step(name):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("from functools import partial\n\ndef test_steps():\n    \"registered steps\"\n    assert sorted(steps) == [\"cut\", \"lower\", \"replace\", \"squash\", \"strip\"]\n    assert steps[\"squash\"](\"  a   b \") == \"a b\" and steps[\"replace\"](\"aXa\", \"X\", \"-\") == \"a-a\" and steps[\"cut\"](\"abcdef\", \"3\") == \"abc\"\n\ndef test_compose():\n    \"composition\"\n    assert compose(strip, lower)(\"  HeLLo \") == \"hello\" and compose()(\"same\") == \"same\"\n\ndef test_build():\n    \"partial and reduce\"\n    clean = build(\"strip|lower|squash|replace:tea:coffee|cut:12\")\n    assert clean(\"   I like   TEA and   tea  \") == \"i like coffe\"\n    assert build(\"lower\")(\"ABC\") == \"abc\" and build(\"replace:a:b|replace:b:c\")(\"aab\") == \"ccc\"\n    try:\n        build(\"strip|explode\")\n        assert False\n    except KeyError:\n        pass\n    assert describe(\"strip|replace:a:b|cut:3\") == \"strip then replace(a, b) then cut(3)\"\n\ndef test_extend():\n    \"registering a new step from outside\"\n    @step(\"title\")\n    def title(text):\n        return text.title()\n    assert build(\"squash|title\")(\" the  long walk \") == \"The Long Walk\"\n    del steps[\"title\"]")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

## Warm-up

If any surprised you, revisit [closures](w060d1.html), [decorators I](w060d2.html), [decorators II](w060d3.html), [the functools module](w060d4.html) or [functional style](w060d5.html).

## Mixed review

## Forge: a plugin host

Write a class `PluginHost()` and two decorator factories:

- `@host.plugin` or `@host.plugin(name=..., priority=...)` registers a function (under its own name unless `name` is given; default priority 0) and returns it **unchanged**. Registering a name twice raises `ValueError`.
- `@host.hook("before")` and `@host.hook("after")` register hook functions; any other kind raises `ValueError` when `hook` is called. Before-hooks are called as `hook(name, args)` before each plugin run, after-hooks as `hook(name, result)` after it, in the order they were registered.
- `host.plugins()`: sorted names. `host.run(name, *args, **kwargs)`: run one plugin with its hooks and return its result (`KeyError` for an unknown name, `RuntimeError` if it's disabled). `host.run_all(*args, **kwargs)`: run every enabled plugin, highest priority first and then by name, returning a dict from name to result in that order. `host.disable(name)` and `host.enable(name)` (`KeyError` if unknown). `host.stats()`: a dict from every plugin's name to how many times it has run **successfully**.
- `timed(clock)`: a decorator factory recording the duration of the last call (by the given clock function) as `wrapper.last` (`None` before any call), even when the call raises.
- `expects(*types)`: a decorator factory checking positional arguments against the types in order, raising `TypeError(f"{name}: argument {i} must be {type_name}")`.

Both factories keep the function's name with `functools.wraps`.

## Forge bonus: cleaning recipes

## Weekly quiz

## Week 60 review

Tick off each one honestly. If one feels shaky, go back and redo two exercises from that day.

- I can explain cells, use nonlocal, avoid the loop trap and choose between a closure and a class. ([Day 1](w060d1.html))
- I can write wrapping and registering decorators with functools.wraps, and predict stacking order. ([Day 2](w060d2.html))
- I can write decorator factories, optional-bracket decorators, class-based and class decorators, and explain property. ([Day 3](w060d3.html))
- I can use cache, lru_cache, partial, reduce, singledispatch and cached_property. ([Day 4](w060d4.html))
- I can write pure functions, update immutable data, and judge when functional style fits. ([Day 5](w060d5.html))
- I built a plugin host and a recipe-driven text cleaner. (Today)

Tomorrow is a **rest day**. Next week: **Context managers**, and what `with` really does.